## Dataset Overview

In [1]:
import pandas as pd

df = pd.read_excel("../data/ApexPlanet_DataAnalytics_Dataset.xlsx")
df.shape

(1000, 12)

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Order_ID       1000 non-null   str    
 1   Order_Date     1000 non-null   str    
 2   Customer_ID    1000 non-null   str    
 3   Customer_Name  1000 non-null   str    
 4   Age            980 non-null    float64
 5   Gender         1000 non-null   str    
 6   City           987 non-null    str    
 7   Product        1000 non-null   str    
 8   Category       1000 non-null   str    
 9   Quantity       1000 non-null   int64  
 10  Unit_Price     1000 non-null   float64
 11  Total_Sales    1000 non-null   float64
dtypes: float64(3), int64(1), str(8)
memory usage: 93.9 KB


### Observation

The dataset contains 1000 records and 12 variables related to retail sales transactions.

The dataset includes customer demographics, product details, and transaction information. Most columns are categorical, while Age, Quantity, Unit_Price, and Total_Sales are numerical variables.

The Order_Date column is currently stored as a string and will require conversion to datetime format for time-based analysis.

## Descriptive Statistics

In [8]:
df.describe()

,Age,Quantity,Unit_Price,Total_Sales
count,980.000000,1000.000000,1000.000000,1000.000000
mean,41.360204,5.435000,25486.783410,139399.439650
std,13.822597,2.838632,14179.402361,114100.051546
min,18.000000,1.000000,145.780000,437.340000
25%,30.000000,3.000000,13895.722500,47066.632500
50%,41.000000,5.000000,25398.740000,108594.025000
75%,54.000000,8.000000,37512.382500,203722.882500
max,65.000000,10.000000,49997.530000,493677.500000


### Observation

The average customer age is approximately 41 years, with ages ranging from 18 to 65 years.

Customers purchase around 5 to 6 units per transaction on average.

The mean Total_Sales value (₹139,399) is higher than the median value (₹108,594), suggesting the presence of high-value transactions that may influence the overall sales distribution.

Further exploratory analysis will be performed to identify sales patterns and potential outliers.

## Data Quality Assessment

In [4]:
df.isnull().sum()

Order_ID          0
Order_Date        0
Customer_ID       0
Customer_Name     0
Age              20
Gender            0
City             13
Product           0
Category          0
Quantity          0
Unit_Price        0
Total_Sales       0
dtype: int64

In [5]:
df.duplicated().sum()

np.int64(0)

### Observation

Missing values were detected in the Age (20 records) and City (13 records) columns.

No duplicate records were found in the dataset, indicating good data consistency.

The amount of missing data is relatively small and can be handled during the data preparation stage.

## Date Format Conversion

In [9]:
df['Order_Date'].head()

0    2025-02-25
1    2025-10-14
2    2025-05-13
3    2025-12-02
4    2025-11-20
Name: Order_Date, dtype: str

In [10]:
df['Order_Date'] = pd.to_datetime(df['Order_Date'])
df['Order_Date'].dtype

dtype('<M8[us]')

The Order_Date column was converted from string format to datetime format to enable time-based analysis such as monthly trends and seasonal sales patterns.

## Handling Missing Values in Age

The Age column contains 20 missing values.

Since age is a numerical variable and may contain extreme values, the median was chosen instead of the mean for imputation. Median is less sensitive to outliers and provides a more robust estimate of the central tendency.

In [15]:
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Age'].isnull().sum()

np.int64(0)

### Observation

All missing values in the Age column were successfully handled using median imputation.

The dataset now contains complete age information while preserving the overall distribution of the variable.

## Handling Missing Values in City

The City column contains 13 missing values.

Since City is a categorical variable and no single city dominates the dataset, replacing missing values with the mode could introduce bias.

Therefore, missing values were replaced with the label "Unknown" to preserve the records while maintaining data integrity.

In [17]:
df['City'].value_counts().head()

City
Patna        135
Kolkata      133
Mumbai       131
Hyderabad    125
Delhi        125
Name: count, dtype: int64

In [19]:
df['City'] = df['City'].fillna('Unknown')
df['City'].isnull().sum()

np.int64(0)

### Observation

All missing values in the City column were successfully handled by assigning the category "Unknown".

This approach avoids introducing assumptions about customer locations and preserves the original dataset size.

## Data Validation

In [23]:
df['Calculated_Sales'] = df['Quantity'] * df['Unit_Price']
(df['Calculated_Sales'] == df['Total_Sales']).sum()

np.int64(822)

In [24]:
len(df)

1000

In [25]:
df[
    df['Calculated_Sales'] != df['Total_Sales']
][[
    'Quantity',
    'Unit_Price',
    'Calculated_Sales',
    'Total_Sales'
]].head(10)

,Quantity,Unit_Price,Calculated_Sales,Total_Sales
7,3,9488.83,28466.49,28466.49
21,7,15396.37,107774.59,107774.59
24,7,36069.86,252489.02,252489.02
27,5,14534.79,72673.95,72673.95
32,5,17556.08,87780.40,87780.40
33,5,46822.84,234114.20,234114.20
42,6,16521.44,99128.64,99128.64
44,9,1491.63,13424.67,13424.67
51,6,4656.10,27936.60,27936.60
58,5,36809.08,184045.40,184045.40


In [26]:
import numpy as np

np.isclose(
    df['Calculated_Sales'],
    df['Total_Sales']
).sum()

np.int64(1000)

### Observation

A validation check was performed to verify whether Total_Sales equals Quantity × Unit_Price.

Direct equality comparison showed minor mismatches due to floating-point precision. A tolerance-based comparison using NumPy's isclose() function confirmed that all 1000 records satisfy the sales calculation rule.

The dataset passed the validation check successfully.

## Feature Engineering

In [27]:
df['Month'] = df['Order_Date'].dt.month_name()
df[['Order_Date', 'Month']].head()

,Order_Date,Month
0,2025-02-25,February
1,2025-10-14,October
2,2025-05-13,May
3,2025-12-02,December
4,2025-11-20,November


A Month column was extracted from Order_Date to support monthly sales analysis and business trend identification.

## Feature Engineering: Age Groups

Customer ages were grouped into meaningful categories to simplify demographic analysis.

Creating age groups enables easier comparison of purchasing behavior across different customer segments and supports business-oriented insights.

In [28]:
bins = [18, 25, 35, 45, 55, 65]

labels = [
    '18-25',
    '26-35',
    '36-45',
    '46-55',
    '56-65'
]

df['Age_Group'] = pd.cut(
    df['Age'],
    bins=bins,
    labels=labels,
    include_lowest=True
)

In [29]:
df[['Age', 'Age_Group']].head()
df['Age_Group'].value_counts()

Age_Group
36-45    223
26-35    219
46-55    202
56-65    198
18-25    158
Name: count, dtype: int64

### Observation

Customers are fairly distributed across different age groups, with the highest representation in the 36–45 and 26–35 segments.

The balanced distribution of age groups indicates that the dataset represents a diverse customer base, making it suitable for demographic and behavioral analysis.

The 18–25 age group has comparatively fewer customers, while the remaining age segments are distributed relatively evenly.